# Homework Starter — Stage 04: Data Acquisition and Ingestion
Name: Doris Lee
Date: 2026-08-18

## Objectives
- API ingestion with secrets in `.env`
- Scrape a permitted public table
- Validate and save raw data to `data/raw/`

In [ ]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install pandas
# !pip install requests
# !pip install yfinance
# !pip install python-dotenv
# !pip install beautifulsoup4

In [ ]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    (".env", "NEEDED", "YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing"),
    (".env.example", "NEEDED", "shipped with this stage - the template you copy to .env"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/dorislee/bootcamp_peiyun_lee/homework/homework4

  [OK ]  NEEDED    .env                                YOU create this: copy .env.example to .env. Missing = no error, but the config demo silently shows nothing
  [OK ]  NEEDED    .env.example                        shipped with this stage - the template you copy to .env

All needed files present.


In [ ]:
import os, pathlib, datetime as dt
import requests
import pandas as pd
from bs4 import BeautifulSoup
from dotenv import load_dotenv

RAW = pathlib.Path('data/raw'); RAW.mkdir(parents=True, exist_ok=True)
load_dotenv(); print('ALPHAVANTAGE_API_KEY loaded?', bool(os.getenv('ALPHAVANTAGE_API_KEY')))

ALPHAVANTAGE_API_KEY loaded? False


## Helpers (use or modify)

In [ ]:
def ts():
    return dt.datetime.now().strftime('%Y%m%d-%H%M%S')

def save_csv(df: pd.DataFrame, prefix: str, **meta):
    mid = '_'.join([f"{k}-{v}" for k,v in meta.items()])
    path = RAW / f"{prefix}_{mid}_{ts()}.csv"
    df.to_csv(path, index=False)
    print('Saved', path)
    return path

def validate(df: pd.DataFrame, required):
    missing = [c for c in required if c not in df.columns]
    return {'missing': missing, 'shape': df.shape, 'na_total': int(df.isna().sum().sum())}

## Part 1 — API Pull (Required)
Choose an endpoint (e.g., Alpha Vantage or use `yfinance` fallback).

In [ ]:
SYMBOL = 'AAPL'
USE_ALPHA = bool(os.getenv('ALPHAVANTAGE_API_KEY'))
if USE_ALPHA:
    url = 'https://www.alphavantage.co/query'
    params = {'function':'TIME_SERIES_DAILY_ADJUSTED','symbol':SYMBOL,'outputsize':'compact','apikey':os.getenv('ALPHAVANTAGE_API_KEY')}
    r = requests.get(url, params=params, timeout=30)
    r.raise_for_status()
    js = r.json()
    key = [k for k in js if 'Time Series' in k][0]
    df_api = pd.DataFrame(js[key]).T.reset_index().rename(columns={'index':'date','5. adjusted close':'adj_close'})[['date','adj_close']]
    df_api['date'] = pd.to_datetime(df_api['date']); df_api['adj_close'] = pd.to_numeric(df_api['adj_close'])
else:
    import yfinance as yf
    # auto_adjust=False keeps the 'Adj Close' column; yfinance 0.2.x returns a
    # MultiIndex of columns for a single ticker, so flatten it before selecting.
    raw = yf.download(SYMBOL, period='3mo', interval='1d', auto_adjust=False, progress=False)
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    df_api = raw.reset_index()[['Date', 'Adj Close']]
    df_api.columns = ['date', 'adj_close']
    df_api['date'] = pd.to_datetime(df_api['date'])
    df_api['adj_close'] = pd.to_numeric(df_api['adj_close'], errors='coerce')

v_api = validate(df_api, ['date', 'adj_close']); v_api

{'missing': [], 'shape': (63, 2), 'na_total': 1}

In [ ]:
_ = save_csv(df_api.sort_values('date'), prefix='api', source='alpha' if USE_ALPHA else 'yfinance', symbol=SYMBOL)

Saved data/raw/api_source-yfinance_symbol-AAPL_20260818-002403.csv


## Part 2 — Scrape a Public Table (Required)
Replace `SCRAPE_URL` with a permitted page containing a simple table.

In [ ]:
SCRAPE_URL = 'https://en.wikipedia.org/wiki/List_of_S%26P_500_companies'  # permitted public table
TABLE_ID  = 'constituents'   # target this specific <table>, not every <tr> on the page
headers = {'User-Agent':'AFE-Homework/1.0 (educational use)'}
try:
    resp = requests.get(SCRAPE_URL, headers=headers, timeout=30); resp.raise_for_status()
    soup = BeautifulSoup(resp.text, 'html.parser')
    table = soup.find('table', id=TABLE_ID)
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in table.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)
    df_scrape.columns = [c.replace('GICSSector', 'GICS Sector') for c in df_scrape.columns]
except Exception as e:
    print('Scrape failed, using inline demo table:', e)
    html = '<table><tr><th>Ticker</th><th>Price</th></tr><tr><td>AAA</td><td>101.2</td></tr></table>'
    soup = BeautifulSoup(html, 'html.parser')
    rows = [[c.get_text(strip=True) for c in tr.find_all(['th','td'])] for tr in soup.find_all('tr')]
    header, *data = [r for r in rows if r]
    df_scrape = pd.DataFrame(data, columns=header)

# Type corrections: 'Founded' (year) → int, 'Date added' → datetime.
# 'CIK' stays text because its leading zeros are meaningful.
if 'Founded' in df_scrape.columns:
    df_scrape['Founded'] = pd.to_numeric(df_scrape['Founded'], errors='coerce').astype('Int64')
if 'Date added' in df_scrape.columns:
    df_scrape['Date added'] = pd.to_datetime(df_scrape['Date added'], errors='coerce')
v_scrape = validate(df_scrape, ['Symbol', 'Security', 'Founded']); v_scrape

{'missing': [], 'shape': (502, 8), 'na_total': 39}

In [ ]:
_ = save_csv(df_scrape, prefix='scrape', site='wikipedia', table='sp500')

Saved data/raw/scrape_site-wikipedia_table-sp500_20260818-002404.csv


## Documentation
- API Source: yfinance (Yahoo Finance) — `yf.download('AAPL', period='3mo', interval='1d', auto_adjust=False)`. If `ALPHAVANTAGE_API_KEY` is set in `.env`, Alpha Vantage `TIME_SERIES_DAILY_ADJUSTED` is used instead.
- Scrape Source: Wikipedia "List of S&P 500 companies" — https://en.wikipedia.org/wiki/List_of_S%26P_500_companies, `<table id="constituents">` (502 rows × 8 columns).
- Assumptions & risks:
  - yfinance can be rate-limited (HTTP 429); retry later or set an API key as fallback.
  - The Wikipedia table selector (`id="constituents"`) is fragile — schema/columns can change.
  - `Founded` (year) and `Date added` are parsed as int / datetime; `CIK` is kept as text because its leading zeros are meaningful.
- Confirm `.env` is not committed: `.env` is listed in the repo root `.gitignore`.